<a href="https://colab.research.google.com/github/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/blob/main/CODIGOS_FUNCIONANDO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from io import BytesIO
from datetime import datetime

# ============================================================================
# PASO 1: DESCARGAR Y CARGAR DATOS
# ============================================================================
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer las hojas necesarias
df_datos = pd.read_excel(excel_file, sheet_name='Datos')
df_forecast = pd.read_excel(excel_file, sheet_name='Forecast')
precios_df = pd.read_excel(excel_file, sheet_name='Precios-Costos')

print("="*80)
print("ANÁLISIS ABC-XYZ COMBINADO")
print("="*80)

# ============================================================================
# PASO 2: ANÁLISIS ABC (Basado en Valor de Ventas)
# ============================================================================
print("\n🔵 CALCULANDO CLASIFICACIÓN ABC...")

# Limpiar nombres de columnas
df_forecast.columns = df_forecast.columns.str.strip()
precios_df.columns = precios_df.columns.str.strip()

# Calcular ventas totales por producto
df_forecast['Ventas_Totales'] = df_forecast.iloc[:, 1:].sum(axis=1)

# Unir con precios
df_abc = df_forecast.merge(precios_df[['Producto', 'PRECIO']], on='Producto', how='left')

# Conversión segura de PRECIO
precio_col = df_abc['PRECIO']
if pd.api.types.is_numeric_dtype(precio_col):
    df_abc['PRECIO'] = pd.to_numeric(precio_col, errors='coerce')
else:
    df_abc['PRECIO'] = pd.to_numeric(
        precio_col.astype(str).str.replace(r'[$,\s]', '', regex=True),
        errors='coerce'
    )

# Eliminar productos sin precio
df_abc = df_abc.dropna(subset=['PRECIO'])

# Calcular Valor Total de Ventas
df_abc['Valor_Total_Ventas'] = df_abc['Ventas_Totales'] * df_abc['PRECIO']

# Ordenar por valor descendente
df_abc = df_abc.sort_values(by='Valor_Total_Ventas', ascending=False).reset_index(drop=True)

# Calcular porcentaje acumulado
total_valor = df_abc['Valor_Total_Ventas'].sum()
df_abc['Porcentaje_Acumulado'] = df_abc['Valor_Total_Ventas'].cumsum() / total_valor * 100

# Clasificación ABC
def classify_abc(porcentaje_acumulado):
    if porcentaje_acumulado <= 60:
        return 'A'
    elif porcentaje_acumulado <= 80:
        return 'B'
    else:
        return 'C'

df_abc['Clasificacion_ABC'] = df_abc['Porcentaje_Acumulado'].apply(classify_abc)

print(f"✅ ABC calculado para {len(df_abc)} productos")

# ============================================================================
# PASO 3: ANÁLISIS XYZ (Basado en Variabilidad)
# ============================================================================
print("\n🟢 CALCULANDO CLASIFICACIÓN XYZ...")

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()

# Renombrar primera columna
primera_col = df_datos.columns[0]
df_datos.rename(columns={primera_col: 'Producto'}, inplace=True)

# Identificar columnas de período
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

# Buscar columna 'oct-25'
def find_oct25_column(columns):
    # Buscar exactamente 'oct-25'
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    # Buscar variantes
    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25']
    for variant in variants:
        if variant in columns:
            return variant

    # Buscar por fecha
    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col
        if isinstance(col, datetime) and col.year == 2025 and col.month == 10:
            return col

    return None

target_col = find_oct25_column(period_cols)

if target_col is None:
    print("⚠️ No se encontró 'oct-25', usando todas las columnas disponibles")
    cols_to_use = period_cols
else:
    idx_target = period_cols.index(target_col)
    cols_to_use = period_cols[:idx_target + 1]

# Extraer datos de ventas
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

# Calcular estadísticas
mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Crear DataFrame XYZ
df_xyz = pd.DataFrame({
    'Producto': cv.index,
    'Media': mean.values,
    'Desv_Std': std.values,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

# Clasificar XYZ por percentiles
p33 = df_xyz['CV'].quantile(0.33)
p67 = df_xyz['CV'].quantile(0.67)

def xyz_class(cv_val):
    if cv_val <= p33:
        return 'X'  # Demanda estable
    elif cv_val <= p67:
        return 'Y'  # Demanda moderada
    else:
        return 'Z'  # Demanda variable

df_xyz['Clasificacion_XYZ'] = df_xyz['CV'].apply(xyz_class)

print(f"✅ XYZ calculado para {len(df_xyz)} productos")

# ============================================================================
# PASO 4: COMBINAR ABC Y XYZ
# ============================================================================
print("\n🟣 COMBINANDO CLASIFICACIONES...")

df_final = df_abc.merge(
    df_xyz[['Producto', 'Media', 'Desv_Std', 'CV', 'Clasificacion_XYZ']],
    on='Producto',
    how='left'
)

# Crear clasificación combinada
df_final['Clasificacion_ABC_XYZ'] = df_final['Clasificacion_ABC'] + '-' + df_final['Clasificacion_XYZ'].fillna('?')

# Ordenar por ABC primero, luego por XYZ
df_final = df_final.sort_values(['Clasificacion_ABC', 'Clasificacion_XYZ']).reset_index(drop=True)

# ============================================================================
# PASO 5: REPORTES Y ANÁLISIS
# ============================================================================
print("\n" + "="*80)
print("RESULTADOS DE CLASIFICACIÓN ABC-XYZ")
print("="*80)

# Resumen ABC
print("\n📊 RESUMEN CLASIFICACIÓN ABC (por Valor):")
abc_summary = df_final.groupby('Clasificacion_ABC').agg({
    'Producto': 'count',
    'Valor_Total_Ventas': 'sum'
}).rename(columns={'Producto': 'Cantidad'})
abc_summary['% del Total'] = (abc_summary['Valor_Total_Ventas'] / total_valor * 100).round(2)
print(abc_summary)

# Resumen XYZ
print("\n📊 RESUMEN CLASIFICACIÓN XYZ (por Variabilidad):")
xyz_summary = df_final['Clasificacion_XYZ'].value_counts().sort_index()
for clase in ['X', 'Y', 'Z']:
    if clase in xyz_summary.index:
        cant = xyz_summary[clase]
        porc = (cant / len(df_final)) * 100
        print(f"   Clase {clase}: {cant:3d} productos ({porc:5.1f}%) - " +
              f"{'Estable' if clase == 'X' else 'Moderada' if clase == 'Y' else 'Variable'}")

# Matriz ABC-XYZ
print("\n📊 MATRIZ ABC-XYZ (Cantidad de productos):")
matriz = pd.crosstab(df_final['Clasificacion_ABC'], df_final['Clasificacion_XYZ'], margins=True)
print(matriz)

# Productos prioritarios (A-X, A-Y, B-X)
print("\n⭐ PRODUCTOS PRIORITARIOS (Alta rotación + Estabilidad):")
prioritarios = df_final[df_final['Clasificacion_ABC_XYZ'].isin(['A-X', 'A-Y', 'B-X'])]
print(f"   Total: {len(prioritarios)} productos")
print(prioritarios[['Producto', 'Clasificacion_ABC_XYZ', 'Valor_Total_Ventas', 'CV']].head(10))

# Productos críticos (A-Z, B-Z)
print("\n⚠️ PRODUCTOS CRÍTICOS (Alta rotación + Alta variabilidad):")
criticos = df_final[df_final['Clasificacion_ABC_XYZ'].isin(['A-Z', 'B-Z'])]
print(f"   Total: {len(criticos)} productos (requieren atención especial)")
print(criticos[['Producto', 'Clasificacion_ABC_XYZ', 'Valor_Total_Ventas', 'CV']].head(10))

# ============================================================================
# PASO 6: EXPORTAR RESULTADOS
# ============================================================================
output_file = 'clasificacion_abc_xyz_combinada.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Hoja principal con todos los datos
    df_final[['Producto', 'Ventas_Totales', 'PRECIO', 'Valor_Total_Ventas',
              'Porcentaje_Acumulado', 'Clasificacion_ABC',
              'Media', 'Desv_Std', 'CV', 'Clasificacion_XYZ',
              'Clasificacion_ABC_XYZ']].to_excel(
        writer, sheet_name='Clasificacion Completa', index=False
    )

    # Matriz ABC-XYZ
    matriz.to_excel(writer, sheet_name='Matriz ABC-XYZ')

    # Resumen ABC
    abc_summary.to_excel(writer, sheet_name='Resumen ABC')

    # Productos prioritarios
    prioritarios[['Producto', 'Clasificacion_ABC_XYZ', 'Valor_Total_Ventas', 'CV']].to_excel(
        writer, sheet_name='Productos Prioritarios', index=False
    )

    # Productos críticos
    criticos[['Producto', 'Clasificacion_ABC_XYZ', 'Valor_Total_Ventas', 'CV']].to_excel(
        writer, sheet_name='Productos Criticos', index=False
    )

print(f"\n✅ Resultados exportados a: {output_file}")
print("\n" + "="*80)
print("ANÁLISIS COMPLETADO")
print("="*80)

ANÁLISIS ABC-XYZ COMBINADO

🔵 CALCULANDO CLASIFICACIÓN ABC...
✅ ABC calculado para 30 productos

🟢 CALCULANDO CLASIFICACIÓN XYZ...
✅ XYZ calculado para 30 productos

🟣 COMBINANDO CLASIFICACIONES...

RESULTADOS DE CLASIFICACIÓN ABC-XYZ

📊 RESUMEN CLASIFICACIÓN ABC (por Valor):
                   Cantidad  Valor_Total_Ventas  % del Total
Clasificacion_ABC                                           
A                        14            330447.2        56.81
B                         7            134736.8        23.16
C                         9            116525.8        20.03

📊 RESUMEN CLASIFICACIÓN XYZ (por Variabilidad):
   Clase X:  10 productos ( 33.3%) - Estable
   Clase Y:  10 productos ( 33.3%) - Moderada
   Clase Z:  10 productos ( 33.3%) - Variable

📊 MATRIZ ABC-XYZ (Cantidad de productos):
Clasificacion_XYZ   X   Y   Z  All
Clasificacion_ABC                 
A                   3   6   5   14
B                   4   2   1    7
C                   3   2   4    9
All           

In [2]:
import pandas as pd
import requests
from io import BytesIO
from datetime import datetime
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PASO 1: DESCARGAR Y CARGAR DATOS
# ============================================================================
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

# Leer las hojas necesarias
df_datos = pd.read_excel(excel_file, sheet_name='Datos')
df_forecast = pd.read_excel(excel_file, sheet_name='Forecast')
precios_df = pd.read_excel(excel_file, sheet_name='Precios-Costos')

print("="*80)
print("ANÁLISIS ABC-XYZ COMBINADO CON FORECAST HOLT-WINTERS")
print("="*80)

# ============================================================================
# PASO 2: ANÁLISIS ABC (Basado en Valor de Ventas)
# ============================================================================
print("\n🔵 CALCULANDO CLASIFICACIÓN ABC...")

# Limpiar nombres de columnas
df_forecast.columns = df_forecast.columns.str.strip()
precios_df.columns = precios_df.columns.str.strip()

# Calcular ventas totales por producto
df_forecast['Ventas_Totales'] = df_forecast.iloc[:, 1:].sum(axis=1)

# Unir con precios
df_abc = df_forecast.merge(precios_df[['Producto', 'PRECIO']], on='Producto', how='left')

# Conversión segura de PRECIO
precio_col = df_abc['PRECIO']
if pd.api.types.is_numeric_dtype(precio_col):
    df_abc['PRECIO'] = pd.to_numeric(precio_col, errors='coerce')
else:
    df_abc['PRECIO'] = pd.to_numeric(
        precio_col.astype(str).str.replace(r'[$,\s]', '', regex=True),
        errors='coerce'
    )

# Eliminar productos sin precio
df_abc = df_abc.dropna(subset=['PRECIO'])

# Calcular Valor Total de Ventas
df_abc['Valor_Total_Ventas'] = df_abc['Ventas_Totales'] * df_abc['PRECIO']

# Ordenar por valor descendente
df_abc = df_abc.sort_values(by='Valor_Total_Ventas', ascending=False).reset_index(drop=True)

# Calcular porcentaje acumulado
total_valor = df_abc['Valor_Total_Ventas'].sum()
df_abc['Porcentaje_Acumulado'] = df_abc['Valor_Total_Ventas'].cumsum() / total_valor * 100

# Clasificación ABC
def classify_abc(porcentaje_acumulado):
    if porcentaje_acumulado <= 60:
        return 'A'
    elif porcentaje_acumulado <= 80:
        return 'B'
    else:
        return 'C'

df_abc['Clasificacion_ABC'] = df_abc['Porcentaje_Acumulado'].apply(classify_abc)

print(f"✅ ABC calculado para {len(df_abc)} productos")

# ============================================================================
# PASO 3: ANÁLISIS XYZ (Basado en Variabilidad)
# ============================================================================
print("\n🟢 CALCULANDO CLASIFICACIÓN XYZ...")

# Limpiar nombres de columnas
df_datos.columns = df_datos.columns.astype(str).str.strip()

# Renombrar primera columna
primera_col = df_datos.columns[0]
df_datos.rename(columns={primera_col: 'Producto'}, inplace=True)

# Identificar columnas de período
all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

# Buscar columna 'oct-25'
def find_oct25_column(columns):
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25']
    for variant in variants:
        if variant in columns:
            return variant

    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col
        if isinstance(col, datetime) and col.year == 2025 and col.month == 10:
            return col

    return None

target_col = find_oct25_column(period_cols)

if target_col is None:
    print("⚠️ No se encontró 'oct-25', usando todas las columnas disponibles")
    cols_to_use = period_cols
else:
    idx_target = period_cols.index(target_col)
    cols_to_use = period_cols[:idx_target + 1]

print(f"📅 Usando {len(cols_to_use)} períodos de datos")

# Extraer datos de ventas
ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

# Convertir a numérico
for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

# Calcular estadísticas
mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

# Crear DataFrame XYZ
df_xyz = pd.DataFrame({
    'Producto': cv.index,
    'Media': mean.values,
    'Desv_Std': std.values,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

# Clasificar XYZ por percentiles
p33 = df_xyz['CV'].quantile(0.33)
p67 = df_xyz['CV'].quantile(0.67)

def xyz_class(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

df_xyz['Clasificacion_XYZ'] = df_xyz['CV'].apply(xyz_class)

print(f"✅ XYZ calculado para {len(df_xyz)} productos")

# ============================================================================
# PASO 4: COMBINAR ABC Y XYZ
# ============================================================================
print("\n🟣 COMBINANDO CLASIFICACIONES...")

df_final = df_abc.merge(
    df_xyz[['Producto', 'Media', 'Desv_Std', 'CV', 'Clasificacion_XYZ']],
    on='Producto',
    how='left'
)

df_final['Clasificacion_ABC_XYZ'] = df_final['Clasificacion_ABC'] + '-' + df_final['Clasificacion_XYZ'].fillna('?')

# ============================================================================
# PASO 5: REPORTES Y ANÁLISIS
# ============================================================================
print("\n" + "="*80)
print("RESULTADOS DE CLASIFICACIÓN ABC-XYZ")
print("="*80)

# Resumen ABC
print("\n📊 RESUMEN CLASIFICACIÓN ABC:")
abc_summary = df_final.groupby('Clasificacion_ABC').agg({
    'Producto': 'count',
    'Valor_Total_Ventas': 'sum'
}).rename(columns={'Producto': 'Cantidad'})
abc_summary['% del Total'] = (abc_summary['Valor_Total_Ventas'] / total_valor * 100).round(2)
print(abc_summary)

# Resumen XYZ
print("\n📊 RESUMEN CLASIFICACIÓN XYZ:")
xyz_summary = df_final['Clasificacion_XYZ'].value_counts().sort_index()
for clase in ['X', 'Y', 'Z']:
    if clase in xyz_summary.index:
        cant = xyz_summary[clase]
        porc = (cant / len(df_final)) * 100
        print(f"   Clase {clase}: {cant:3d} productos ({porc:5.1f}%)")

# Matriz ABC-XYZ
print("\n📊 MATRIZ ABC-XYZ:")
matriz = pd.crosstab(df_final['Clasificacion_ABC'], df_final['Clasificacion_XYZ'], margins=True)
print(matriz)

# ============================================================================
# PASO 6: FORECAST HOLT-WINTERS PARA PRODUCTOS A-X
# ============================================================================
print("\n" + "="*80)
print("FORECAST HOLT-WINTERS PARA PRODUCTOS A-X")
print("="*80)

# Filtrar productos A-X
productos_ax = df_final[df_final['Clasificacion_ABC_XYZ'] == 'A-X']['Producto'].tolist()

print(f"\n📈 Analizando {len(productos_ax)} productos A-X (Alta rotación + Demanda estable)")
print(f"Períodos de entrenamiento: {len(cols_to_use)}")
print(f"Horizonte de forecast: 18 meses\n")

if len(productos_ax) == 0:
    print("⚠️ No hay productos clasificados como A-X")
else:
    resultados_forecast = []

    for i, producto in enumerate(productos_ax, 1):
        try:
            # Obtener serie temporal del producto
            serie = ventas.loc[producto, cols_to_use].values

            # Verificar que hay suficientes datos
            if len(serie) < 12 or np.sum(serie > 0) < 8:
                print(f"⚠️ [{i}/{len(productos_ax)}] {producto}: Datos insuficientes")
                continue

            # Dividir en train/test (últimos 6 meses para validación)
            train_size = len(serie) - 6
            train = serie[:train_size]
            test = serie[train_size:]

            # ===== MODELO 1: HOLT (Doble suavización - sin estacionalidad) =====
            try:
                model_holt = ExponentialSmoothing(
                    train,
                    trend='add',
                    seasonal=None,
                    initialization_method='estimated'
                ).fit(optimized=True)

                forecast_holt = model_holt.forecast(steps=len(test))
                rmse_holt = np.sqrt(mean_squared_error(test, forecast_holt))
                alpha_holt = model_holt.params['smoothing_level']
                beta_holt = model_holt.params['smoothing_trend']

            except Exception as e:
                rmse_holt = float('inf')
                alpha_holt = None
                beta_holt = None

            # ===== MODELO 2: HOLT-WINTERS (Triple suavización - con estacionalidad) =====
            try:
                # Intentar con estacionalidad de 12 meses (si hay suficientes datos)
                seasonal_periods = 12 if len(train) >= 24 else 6

                model_hw = ExponentialSmoothing(
                    train,
                    trend='add',
                    seasonal='add',
                    seasonal_periods=seasonal_periods,
                    initialization_method='estimated'
                ).fit(optimized=True)

                forecast_hw = model_hw.forecast(steps=len(test))
                rmse_hw = np.sqrt(mean_squared_error(test, forecast_hw))
                alpha_hw = model_hw.params['smoothing_level']
                beta_hw = model_hw.params['smoothing_trend']
                gamma_hw = model_hw.params['smoothing_seasonal']

            except Exception as e:
                rmse_hw = float('inf')
                alpha_hw = None
                beta_hw = None
                gamma_hw = None

            # Determinar mejor modelo
            if rmse_holt < rmse_hw:
                mejor_modelo = 'Holt'
                mejor_rmse = rmse_holt
                # Forecast 18 meses con mejor modelo
                model_final = ExponentialSmoothing(
                    serie,
                    trend='add',
                    seasonal=None,
                    initialization_method='estimated'
                ).fit(optimized=True)
                forecast_18 = model_final.forecast(steps=18)
                params = f"α={alpha_holt:.4f}, β={beta_holt:.4f}"
            else:
                mejor_modelo = 'Holt-Winters'
                mejor_rmse = rmse_hw
                # Forecast 18 meses con mejor modelo
                seasonal_periods = 12 if len(serie) >= 24 else 6
                model_final = ExponentialSmoothing(
                    serie,
                    trend='add',
                    seasonal='add',
                    seasonal_periods=seasonal_periods,
                    initialization_method='estimated'
                ).fit(optimized=True)
                forecast_18 = model_final.forecast(steps=18)
                params = f"α={alpha_hw:.4f}, β={beta_hw:.4f}, γ={gamma_hw:.4f}"

            # Guardar resultados
            resultados_forecast.append({
                'Producto': producto,
                'Mejor_Modelo': mejor_modelo,
                'RMSE': mejor_rmse,
                'Parametros': params,
                'RMSE_Holt': rmse_holt if rmse_holt != float('inf') else None,
                'RMSE_HW': rmse_hw if rmse_hw != float('inf') else None,
                'Forecast_Promedio': forecast_18.mean(),
                'Forecast_Total': forecast_18.sum(),
                'Forecast_18M': forecast_18
            })

            print(f"✅ [{i}/{len(productos_ax)}] {producto:30s} | Mejor: {mejor_modelo:15s} | RMSE: {mejor_rmse:8.2f} | {params}")

        except Exception as e:
            print(f"❌ [{i}/{len(productos_ax)}] {producto}: Error - {str(e)[:50]}")

    # ============================================================================
    # PASO 7: RESUMEN DE FORECASTS
    # ============================================================================
    if resultados_forecast:
        df_resultados = pd.DataFrame(resultados_forecast)

        print("\n" + "="*80)
        print("RESUMEN DE FORECASTS - PRODUCTOS A-X")
        print("="*80)

        print(f"\n📊 PRODUCTOS ANALIZADOS: {len(df_resultados)}")

        # Resumen por modelo
        print("\n🔍 MEJOR MODELO POR PRODUCTO:")
        modelo_counts = df_resultados['Mejor_Modelo'].value_counts()
        for modelo, count in modelo_counts.items():
            pct = (count / len(df_resultados)) * 100
            print(f"   {modelo}: {count} productos ({pct:.1f}%)")

        # Top 10 mejor precisión
        print("\n🎯 TOP 10 PRODUCTOS CON MEJOR PRECISIÓN (menor RMSE):")
        top10_rmse = df_resultados.nsmallest(10, 'RMSE')[['Producto', 'Mejor_Modelo', 'RMSE', 'Parametros']]
        print(top10_rmse.to_string(index=False))

        # Estadísticas de RMSE
        print("\n📈 ESTADÍSTICAS DE PRECISIÓN:")
        print(f"   RMSE Promedio: {df_resultados['RMSE'].mean():,.2f}")
        print(f"   RMSE Mediana:  {df_resultados['RMSE'].median():,.2f}")
        print(f"   RMSE Mínimo:   {df_resultados['RMSE'].min():,.2f}")
        print(f"   RMSE Máximo:   {df_resultados['RMSE'].max():,.2f}")

        # Forecast agregado
        print("\n📊 FORECAST AGREGADO PRÓXIMOS 18 MESES:")
        total_forecast = df_resultados['Forecast_Total'].sum()
        promedio_mensual = df_resultados['Forecast_Promedio'].sum()
        print(f"   Total proyectado: {total_forecast:,.0f} unidades")
        print(f"   Promedio mensual: {promedio_mensual:,.0f} unidades")

        # Detalle de algunos productos
        print("\n📋 DETALLE DE FORECAST - TOP 5 PRODUCTOS:")
        for idx, row in df_resultados.nsmallest(5, 'RMSE').iterrows():
            print(f"\n   🔸 {row['Producto']}")
            print(f"      Modelo: {row['Mejor_Modelo']} | RMSE: {row['RMSE']:.2f}")
            print(f"      Parámetros: {row['Parametros']}")
            forecast_vals = row['Forecast_18M']
            print(f"      Forecast (primeros 6 meses): {', '.join([f'{v:.0f}' for v in forecast_vals[:6]])}")
    else:
        print("\n⚠️ No se pudieron generar forecasts para ningún producto A-X")

print("\n" + "="*80)
print("ANÁLISIS COMPLETADO")
print("="*80)

ANÁLISIS ABC-XYZ COMBINADO CON FORECAST HOLT-WINTERS

🔵 CALCULANDO CLASIFICACIÓN ABC...
✅ ABC calculado para 30 productos

🟢 CALCULANDO CLASIFICACIÓN XYZ...
📅 Usando 34 períodos de datos
✅ XYZ calculado para 30 productos

🟣 COMBINANDO CLASIFICACIONES...

RESULTADOS DE CLASIFICACIÓN ABC-XYZ

📊 RESUMEN CLASIFICACIÓN ABC:
                   Cantidad  Valor_Total_Ventas  % del Total
Clasificacion_ABC                                           
A                        14            330447.2        56.81
B                         7            134736.8        23.16
C                         9            116525.8        20.03

📊 RESUMEN CLASIFICACIÓN XYZ:
   Clase X:  10 productos ( 33.3%)
   Clase Y:  10 productos ( 33.3%)
   Clase Z:  10 productos ( 33.3%)

📊 MATRIZ ABC-XYZ:
Clasificacion_XYZ   X   Y   Z  All
Clasificacion_ABC                 
A                   3   6   5   14
B                   4   2   1    7
C                   3   2   4    9
All                10  10  10   30

FORECAST 

In [3]:
import pandas as pd
import requests
from io import BytesIO
from datetime import datetime
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FUNCIONES AUXILIARES - MÉTODO DE CROSTON
# ============================================================================
def croston_forecast(ts, h=18, alpha=0.1):
    """
    Implementación del método de Croston para demanda intermitente

    Parámetros:
    - ts: Serie temporal histórica
    - h: Horizonte de forecast
    - alpha: Parámetro de suavización (se optimizará)

    Retorna:
    - forecast: Predicciones para h períodos
    - alpha_opt: Alpha óptimo utilizado
    """
    ts = np.array(ts)
    n = len(ts)

    # Inicializar demanda y intervalos
    demand = []  # Tamaño de demanda cuando ocurre
    intervals = []  # Períodos entre demandas

    last_demand_idx = 0

    for i in range(n):
        if ts[i] > 0:
            demand.append(ts[i])
            if len(demand) > 1:
                intervals.append(i - last_demand_idx)
            last_demand_idx = i

    if len(demand) < 2:
        # Si hay muy pocas demandas, usar promedio simple
        return np.full(h, ts[ts > 0].mean() if np.any(ts > 0) else 0), alpha

    # Suavización exponencial para demanda e intervalos
    z_smooth = [demand[0]]  # Demanda suavizada
    x_smooth = [intervals[0] if len(intervals) > 0 else 1]  # Intervalo suavizado

    for i in range(1, len(demand)):
        z_smooth.append(alpha * demand[i] + (1 - alpha) * z_smooth[-1])
        if i < len(intervals):
            x_smooth.append(alpha * intervals[i] + (1 - alpha) * x_smooth[-1])

    # Forecast = demanda promedio / intervalo promedio
    forecast_value = z_smooth[-1] / x_smooth[-1] if x_smooth[-1] > 0 else z_smooth[-1]

    return np.full(h, forecast_value), alpha

def optimize_croston_alpha(train, test):
    """
    Optimiza el parámetro alpha de Croston minimizando RMSE
    """
    best_alpha = 0.1
    best_rmse = float('inf')

    for alpha in np.arange(0.05, 0.95, 0.05):
        try:
            forecast, _ = croston_forecast(train, h=len(test), alpha=alpha)
            rmse = np.sqrt(mean_squared_error(test, forecast))
            if rmse < best_rmse:
                best_rmse = rmse
                best_alpha = alpha
        except:
            continue

    return best_alpha, best_rmse

# ============================================================================
# PASO 1: DESCARGAR Y CARGAR DATOS
# ============================================================================
url = "https://github.com/santiagonajera/MODELACION-Y-PRONOSTICOS-DE-LA-DEMANDA/raw/refs/heads/main/EjercicioIntegral-Forecast.xlsx"
response = requests.get(url)
excel_file = BytesIO(response.content)

df_datos = pd.read_excel(excel_file, sheet_name='Datos')
df_forecast = pd.read_excel(excel_file, sheet_name='Forecast')
precios_df = pd.read_excel(excel_file, sheet_name='Precios-Costos')

print("="*90)
print("ANÁLISIS ABC-XYZ CON FORECAST OPTIMIZADO (HOLT, HOLT-WINTERS Y CROSTON)")
print("="*90)

# ============================================================================
# PASO 2: ANÁLISIS ABC
# ============================================================================
print("\n🔵 CALCULANDO CLASIFICACIÓN ABC...")

df_forecast.columns = df_forecast.columns.str.strip()
precios_df.columns = precios_df.columns.str.strip()

df_forecast['Ventas_Totales'] = df_forecast.iloc[:, 1:].sum(axis=1)
df_abc = df_forecast.merge(precios_df[['Producto', 'PRECIO']], on='Producto', how='left')

precio_col = df_abc['PRECIO']
if pd.api.types.is_numeric_dtype(precio_col):
    df_abc['PRECIO'] = pd.to_numeric(precio_col, errors='coerce')
else:
    df_abc['PRECIO'] = pd.to_numeric(
        precio_col.astype(str).str.replace(r'[$,\s]', '', regex=True),
        errors='coerce'
    )

df_abc = df_abc.dropna(subset=['PRECIO'])
df_abc['Valor_Total_Ventas'] = df_abc['Ventas_Totales'] * df_abc['PRECIO']
df_abc = df_abc.sort_values(by='Valor_Total_Ventas', ascending=False).reset_index(drop=True)

total_valor = df_abc['Valor_Total_Ventas'].sum()
df_abc['Porcentaje_Acumulado'] = df_abc['Valor_Total_Ventas'].cumsum() / total_valor * 100

def classify_abc(porcentaje_acumulado):
    if porcentaje_acumulado <= 60:
        return 'A'
    elif porcentaje_acumulado <= 80:
        return 'B'
    else:
        return 'C'

df_abc['Clasificacion_ABC'] = df_abc['Porcentaje_Acumulado'].apply(classify_abc)
print(f"✅ ABC calculado para {len(df_abc)} productos")

# ============================================================================
# PASO 3: ANÁLISIS XYZ
# ============================================================================
print("\n🟢 CALCULANDO CLASIFICACIÓN XYZ...")

df_datos.columns = df_datos.columns.astype(str).str.strip()
primera_col = df_datos.columns[0]
df_datos.rename(columns={primera_col: 'Producto'}, inplace=True)

all_cols = df_datos.columns.tolist()
period_cols = [col for col in all_cols if col != 'Producto']

def find_oct25_column(columns):
    for col in columns:
        if str(col).lower().strip() == 'oct-25':
            return col

    variants = ['oct-25', 'Oct-25', 'oct 25', 'Oct 25']
    for variant in variants:
        if variant in columns:
            return variant

    for col in columns:
        col_str = str(col).lower()
        if '2025' in col_str and '10' in col_str:
            return col
        if isinstance(col, datetime) and col.year == 2025 and col.month == 10:
            return col

    return None

target_col = find_oct25_column(period_cols)

if target_col is None:
    print("⚠️ No se encontró 'oct-25', usando todas las columnas disponibles")
    cols_to_use = period_cols
else:
    idx_target = period_cols.index(target_col)
    cols_to_use = period_cols[:idx_target + 1]

print(f"📅 Usando {len(cols_to_use)} períodos de datos")

ventas = df_datos[['Producto'] + cols_to_use].copy()
ventas.set_index('Producto', inplace=True)

for col in ventas.columns:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(how='all')

mean = ventas.mean(axis=1)
std = ventas.std(axis=1)
cv = (std / mean).replace([float('inf'), -float('inf')], pd.NA)

df_xyz = pd.DataFrame({
    'Producto': cv.index,
    'Media': mean.values,
    'Desv_Std': std.values,
    'CV': cv.values
}).dropna(subset=['CV']).reset_index(drop=True)

p33 = df_xyz['CV'].quantile(0.33)
p67 = df_xyz['CV'].quantile(0.67)

def xyz_class(cv_val):
    if cv_val <= p33:
        return 'X'
    elif cv_val <= p67:
        return 'Y'
    else:
        return 'Z'

df_xyz['Clasificacion_XYZ'] = df_xyz['CV'].apply(xyz_class)
print(f"✅ XYZ calculado para {len(df_xyz)} productos")

# ============================================================================
# PASO 4: COMBINAR ABC Y XYZ
# ============================================================================
print("\n🟣 COMBINANDO CLASIFICACIONES...")

df_final = df_abc.merge(
    df_xyz[['Producto', 'Media', 'Desv_Std', 'CV', 'Clasificacion_XYZ']],
    on='Producto',
    how='left'
)

df_final['Clasificacion_ABC_XYZ'] = df_final['Clasificacion_ABC'] + '-' + df_final['Clasificacion_XYZ'].fillna('?')

# ============================================================================
# PASO 5: REPORTES ABC-XYZ
# ============================================================================
print("\n" + "="*90)
print("RESULTADOS DE CLASIFICACIÓN ABC-XYZ")
print("="*90)

print("\n📊 RESUMEN CLASIFICACIÓN ABC:")
abc_summary = df_final.groupby('Clasificacion_ABC').agg({
    'Producto': 'count',
    'Valor_Total_Ventas': 'sum'
}).rename(columns={'Producto': 'Cantidad'})
abc_summary['% del Total'] = (abc_summary['Valor_Total_Ventas'] / total_valor * 100).round(2)
print(abc_summary)

print("\n📊 RESUMEN CLASIFICACIÓN XYZ:")
xyz_summary = df_final['Clasificacion_XYZ'].value_counts().sort_index()
for clase in ['X', 'Y', 'Z']:
    if clase in xyz_summary.index:
        cant = xyz_summary[clase]
        porc = (cant / len(df_final)) * 100
        print(f"   Clase {clase}: {cant:3d} productos ({porc:5.1f}%)")

print("\n📊 MATRIZ ABC-XYZ:")
matriz = pd.crosstab(df_final['Clasificacion_ABC'], df_final['Clasificacion_XYZ'], margins=True)
print(matriz)

# ============================================================================
# PASO 6: FORECAST PARA PRODUCTOS A-X CON 3 MÉTODOS
# ============================================================================
print("\n" + "="*90)
print("FORECAST CON OPTIMIZACIÓN - PRODUCTOS A-X")
print("Comparación: Holt vs Holt-Winters vs Croston")
print("="*90)

productos_ax = df_final[df_final['Clasificacion_ABC_XYZ'] == 'A-X']['Producto'].tolist()

print(f"\n📈 Analizando {len(productos_ax)} productos A-X")
print(f"📅 Períodos de entrenamiento: {len(cols_to_use)}")
print(f"📅 Períodos de validación: 18 meses")
print(f"📅 Horizonte de forecast: 18 meses\n")

if len(productos_ax) == 0:
    print("⚠️ No hay productos clasificados como A-X")
else:
    resultados_forecast = []

    for i, producto in enumerate(productos_ax, 1):
        try:
            # Obtener serie temporal
            serie = ventas.loc[producto, cols_to_use].values

            # Verificar datos suficientes
            if len(serie) < 24:  # Mínimo 24 meses
                print(f"⚠️ [{i}/{len(productos_ax)}] {producto}: Datos insuficientes (< 24 meses)")
                continue

            # Dividir en train/test (últimos 18 meses para validación)
            train_size = len(serie) - 18
            train = serie[:train_size]
            test = serie[train_size:]

            # Calcular intermitencia (% de períodos con demanda cero)
            intermitencia = (serie == 0).sum() / len(serie) * 100

            resultados_modelos = {}

            # ===== MODELO 1: HOLT (α, β óptimos) =====
            try:
                model_holt = ExponentialSmoothing(
                    train,
                    trend='add',
                    seasonal=None,
                    initialization_method='estimated'
                ).fit(optimized=True)

                forecast_holt_val = model_holt.forecast(steps=len(test))
                rmse_holt = np.sqrt(mean_squared_error(test, forecast_holt_val))

                # Parámetros óptimos
                alpha_holt = model_holt.params['smoothing_level']
                beta_holt = model_holt.params['smoothing_trend']

                # Forecast 18 meses con serie completa
                model_holt_full = ExponentialSmoothing(
                    serie,
                    trend='add',
                    seasonal=None,
                    initialization_method='estimated'
                ).fit(optimized=True)
                forecast_holt_18 = model_holt_full.forecast(steps=18)

                resultados_modelos['Holt'] = {
                    'rmse': rmse_holt,
                    'params': f"α={alpha_holt:.4f}, β={beta_holt:.4f}",
                    'forecast': forecast_holt_18,
                    'alpha': alpha_holt,
                    'beta': beta_holt,
                    'gamma': None
                }

            except Exception as e:
                resultados_modelos['Holt'] = {
                    'rmse': float('inf'),
                    'params': 'Error',
                    'forecast': None,
                    'alpha': None,
                    'beta': None,
                    'gamma': None
                }

            # ===== MODELO 2: HOLT-WINTERS (α, β, γ óptimos) =====
            try:
                seasonal_periods = 12 if len(train) >= 36 else 6

                model_hw = ExponentialSmoothing(
                    train,
                    trend='add',
                    seasonal='add',
                    seasonal_periods=seasonal_periods,
                    initialization_method='estimated'
                ).fit(optimized=True)

                forecast_hw_val = model_hw.forecast(steps=len(test))
                rmse_hw = np.sqrt(mean_squared_error(test, forecast_hw_val))

                # Parámetros óptimos
                alpha_hw = model_hw.params['smoothing_level']
                beta_hw = model_hw.params['smoothing_trend']
                gamma_hw = model_hw.params['smoothing_seasonal']

                # Forecast 18 meses con serie completa
                model_hw_full = ExponentialSmoothing(
                    serie,
                    trend='add',
                    seasonal='add',
                    seasonal_periods=seasonal_periods,
                    initialization_method='estimated'
                ).fit(optimized=True)
                forecast_hw_18 = model_hw_full.forecast(steps=18)

                resultados_modelos['Holt-Winters'] = {
                    'rmse': rmse_hw,
                    'params': f"α={alpha_hw:.4f}, β={beta_hw:.4f}, γ={gamma_hw:.4f}",
                    'forecast': forecast_hw_18,
                    'alpha': alpha_hw,
                    'beta': beta_hw,
                    'gamma': gamma_hw
                }

            except Exception as e:
                resultados_modelos['Holt-Winters'] = {
                    'rmse': float('inf'),
                    'params': 'Error',
                    'forecast': None,
                    'alpha': None,
                    'beta': None,
                    'gamma': None
                }

            # ===== MODELO 3: CROSTON (α óptimo) =====
            try:
                # Optimizar alpha de Croston
                alpha_croston_opt, rmse_croston = optimize_croston_alpha(train, test)

                # Forecast 18 meses con serie completa
                forecast_croston_18, _ = croston_forecast(serie, h=18, alpha=alpha_croston_opt)

                resultados_modelos['Croston'] = {
                    'rmse': rmse_croston,
                    'params': f"α={alpha_croston_opt:.4f}",
                    'forecast': forecast_croston_18,
                    'alpha': alpha_croston_opt,
                    'beta': None,
                    'gamma': None
                }

            except Exception as e:
                resultados_modelos['Croston'] = {
                    'rmse': float('inf'),
                    'params': 'Error',
                    'forecast': None,
                    'alpha': None,
                    'beta': None,
                    'gamma': None
                }

            # Determinar mejor modelo
            mejor_modelo = min(resultados_modelos.items(), key=lambda x: x[1]['rmse'])
            nombre_mejor = mejor_modelo[0]
            datos_mejor = mejor_modelo[1]

            # Guardar resultados
            resultado = {
                'Producto': producto,
                'Intermitencia_%': intermitencia,
                'Mejor_Modelo': nombre_mejor,
                'RMSE_Mejor': datos_mejor['rmse'],
                'Parametros_Mejor': datos_mejor['params'],
                'RMSE_Holt': resultados_modelos['Holt']['rmse'],
                'RMSE_HW': resultados_modelos['Holt-Winters']['rmse'],
                'RMSE_Croston': resultados_modelos['Croston']['rmse'],
                'Params_Holt': resultados_modelos['Holt']['params'],
                'Params_HW': resultados_modelos['Holt-Winters']['params'],
                'Params_Croston': resultados_modelos['Croston']['params'],
                'Forecast_18M': datos_mejor['forecast'],
                'Forecast_Promedio': datos_mejor['forecast'].mean() if datos_mejor['forecast'] is not None else 0,
                'Forecast_Total': datos_mejor['forecast'].sum() if datos_mejor['forecast'] is not None else 0
            }

            resultados_forecast.append(resultado)

            # Mostrar progreso
            rmse_str = f"Holt:{resultados_modelos['Holt']['rmse']:7.1f} | " \
                      f"HW:{resultados_modelos['Holt-Winters']['rmse']:7.1f} | " \
                      f"Croston:{resultados_modelos['Croston']['rmse']:7.1f}"

            print(f"✅ [{i:2d}/{len(productos_ax)}] {producto:25s} | Mejor: {nombre_mejor:13s} | {rmse_str}")

        except Exception as e:
            print(f"❌ [{i:2d}/{len(productos_ax)}] {producto}: Error general - {str(e)[:40]}")

    # ============================================================================
    # PASO 7: ANÁLISIS DE RESULTADOS
    # ============================================================================
    if resultados_forecast:
        df_resultados = pd.DataFrame(resultados_forecast)

        print("\n" + "="*90)
        print("RESUMEN COMPARATIVO DE MODELOS")
        print("="*90)

        print(f"\n📊 PRODUCTOS ANALIZADOS: {len(df_resultados)}")

        # Resumen por modelo ganador
        print("\n🏆 MEJOR MODELO POR PRODUCTO:")
        modelo_counts = df_resultados['Mejor_Modelo'].value_counts()
        for modelo, count in modelo_counts.items():
            pct = (count / len(df_resultados)) * 100
            rmse_avg = df_resultados[df_resultados['Mejor_Modelo'] == modelo]['RMSE_Mejor'].mean()
            print(f"   {modelo:15s}: {count:3d} productos ({pct:5.1f}%) - RMSE promedio: {rmse_avg:,.2f}")

        # Comparación de RMSE promedio por método
        print("\n📊 RMSE PROMEDIO POR MÉTODO:")
        rmse_holt_avg = df_resultados[df_resultados['RMSE_Holt'] != float('inf')]['RMSE_Holt'].mean()
        rmse_hw_avg = df_resultados[df_resultados['RMSE_HW'] != float('inf')]['RMSE_HW'].mean()
        rmse_croston_avg = df_resultados[df_resultados['RMSE_Croston'] != float('inf')]['RMSE_Croston'].mean()

        print(f"   Holt:         {rmse_holt_avg:,.2f}")
        print(f"   Holt-Winters: {rmse_hw_avg:,.2f}")
        print(f"   Croston:      {rmse_croston_avg:,.2f}")

        # Intermitencia vs Modelo
        print("\n🔍 RELACIÓN INTERMITENCIA - MEJOR MODELO:")
        for modelo in ['Holt', 'Holt-Winters', 'Croston']:
            df_modelo = df_resultados[df_resultados['Mejor_Modelo'] == modelo]
            if len(df_modelo) > 0:
                intermitencia_avg = df_modelo['Intermitencia_%'].mean()
                print(f"   {modelo:15s}: Intermitencia promedio = {intermitencia_avg:.1f}%")

        # Top 10 mejor precisión
        print("\n🎯 TOP 10 PRODUCTOS CON MEJOR PRECISIÓN:")
        top10 = df_resultados.nsmallest(10, 'RMSE_Mejor')[
            ['Producto', 'Mejor_Modelo', 'RMSE_Mejor', 'Intermitencia_%', 'Parametros_Mejor']
        ]
        print(top10.to_string(index=False))

        # Forecast agregado
        print("\n📈 FORECAST AGREGADO PRÓXIMOS 18 MESES:")
        total_forecast = df_resultados['Forecast_Total'].sum()
        promedio_mensual = df_resultados['Forecast_Promedio'].sum()
        print(f"   Total proyectado:  {total_forecast:,.0f} unidades")
        print(f"   Promedio mensual:  {promedio_mensual:,.0f} unidades/mes")

        # Detalle top 3
        print("\n📋 DETALLE FORECAST - TOP 3 PRODUCTOS (mejor RMSE):")
        for idx, row in df_resultados.nsmallest(3, 'RMSE_Mejor').iterrows():
            print(f"\n   {'='*85}")
            print(f"   🔸 {row['Producto']}")
            print(f"      Modelo seleccionado: {row['Mejor_Modelo']}")
            print(f"      Parámetros: {row['Parametros_Mejor']}")
            print(f"      RMSE: {row['RMSE_Mejor']:.2f} | Intermitencia: {row['Intermitencia_%']:.1f}%")
            print(f"\n      Comparación RMSE:")
            print(f"         Holt:         {row['RMSE_Holt']:8.2f} ({row['Params_Holt']})")
            print(f"         Holt-Winters: {row['RMSE_HW']:8.2f} ({row['Params_HW']})")
            print(f"         Croston:      {row['RMSE_Croston']:8.2f} ({row['Params_Croston']})")

            if row['Forecast_18M'] is not None:
                forecast_vals = row['Forecast_18M']
                print(f"\n      Forecast próximos 18 meses:")
                print(f"         Meses 1-6:   {', '.join([f'{v:6.0f}' for v in forecast_vals[:6]])}")
                print(f"         Meses 7-12:  {', '.join([f'{v:6.0f}' for v in forecast_vals[6:12]])}")
                print(f"         Meses 13-18: {', '.join([f'{v:6.0f}' for v in forecast_vals[12:18]])}")
                print(f"         Total 18M:   {forecast_vals.sum():,.0f} unidades")

    else:
        print("\n⚠️ No se pudieron generar forecasts para ningún producto A-X")

print("\n" + "="*90)
print("ANÁLISIS COMPLETADO")
print("="*90)

ANÁLISIS ABC-XYZ CON FORECAST OPTIMIZADO (HOLT, HOLT-WINTERS Y CROSTON)

🔵 CALCULANDO CLASIFICACIÓN ABC...
✅ ABC calculado para 30 productos

🟢 CALCULANDO CLASIFICACIÓN XYZ...
📅 Usando 34 períodos de datos
✅ XYZ calculado para 30 productos

🟣 COMBINANDO CLASIFICACIONES...

RESULTADOS DE CLASIFICACIÓN ABC-XYZ

📊 RESUMEN CLASIFICACIÓN ABC:
                   Cantidad  Valor_Total_Ventas  % del Total
Clasificacion_ABC                                           
A                        14            330447.2        56.81
B                         7            134736.8        23.16
C                         9            116525.8        20.03

📊 RESUMEN CLASIFICACIÓN XYZ:
   Clase X:  10 productos ( 33.3%)
   Clase Y:  10 productos ( 33.3%)
   Clase Z:  10 productos ( 33.3%)

📊 MATRIZ ABC-XYZ:
Clasificacion_XYZ   X   Y   Z  All
Clasificacion_ABC                 
A                   3   6   5   14
B                   4   2   1    7
C                   3   2   4    9
All                10  10 